# 第 1 阶段：兵器库储备 (PyTorch 与 Hugging Face 基础)

大语言模型 (LLM) 看似神秘莫测，但剖开华丽的表象，其底层不过是极其庞大但规则简单的**矩阵乘法**。而在人工智能领域，处理矩阵的终极兵器就是 **PyTorch**。

在真正解剖 Transformer 之前，我们必须熟练掌握两把基本的手术刀：
1. **PyTorch 张量 (Tensor)**：大模型的血液和骨骼。
2. **Tokenizer (分词器)**：人类语言与矩阵数字交互的翻译官。
3. **量化魔法 (Quantization)**：把庞然大物压缩进平民显卡的黑科技。

---
## 兵器一：PyTorch 与显存转移

### 0. 课前准备 (极其重要！)

**请注意！Colab 的每一个 Notebook 实例都是独立的“虚拟机”。** 
你在上一个 Notebook (如 `00`) 中安装的包、挂载的 Google Drive，在打开新的 Notebook 时都会统统归零。

所以在每一次开启新的挑战前，请养成好习惯，执行以下单元格把神级兵器和硬盘装配好！

In [ ]:
# 1. 安装我们在解剖过程中必需的屠龙刀
# 包括前沿的 transformers 从而支持最新的视觉多模态大模型框架
!pip install -q -U torch transformers accelerate datasets huggingface_hub bitsandbytes qwen-vl-utils git+https://github.com/huggingface/transformers.git

In [ ]:
from google.colab import drive
import os

# 2. 挂载持久化硬盘
drive.mount('/content/drive')

# 3. 指定之前我们在 00 阶段下载模型的地方，这样接下来的黑科技就不会再重复下载庞大的 10GB 权重文件了！

print("✅ 工作区与缓存目录已绑定完毕，可以开始解剖！")

### 1.1 万物皆张量 (Tensor)
在 PyTorch 中，所有的数据——无论是文本、图片还是模型的参数——全都被定义为张量 (Tensor)。它和 Numpy 的多维数组极为相似，但它拥有一个超级能力：**可以在 GPU 上进行闪电般的并行计算**。

In [ ]:
import torch

# 创建一个简单的 2x3 张量（你可以把它看作二维矩阵）
tensor_a = torch.tensor([
    [1.0, 2.0, 3.0], 
    [4.0, 5.0, 6.0]
])

print("张量数据:\n", tensor_a)
print("\n张量维度 (Shape):", tensor_a.shape)   # 极度重要！以后 debug 70% 的时间在对 shape
print("数据类型 (Dtype):", tensor_a.dtype)   # FP32 (float32) 是默认精度
print("所在设备 (Device):", tensor_a.device) # 默认创建在内存 (CPU) 中

### 1.2 设备转移：显存报错的根源
初学者最常碰到的报错叫 `RuntimeError: Expected all tensors to be on the same device`。这是因为**在内存中的张量和在显存中的张量是不能直接相乘的**。

In [ ]:
# 把张量发射到 GPU 上
if torch.cuda.is_available():
    tensor_gpu = tensor_a.to("cuda")
    print("转移后所在设备:", tensor_gpu.device)
else:
    print("没有检测到 GPU，你是不是忘了在 Colab 开启 T4 硬件加速？")

### 1.3 极客加餐：反向传播 (Autograd) 体验
大模型为什么具有“学习能力”？一切魔法的根源是微积分的**链式求导法则**。PyTorch 提供了一个叫做 Autograd 的作弊器，它可以自动计算偏导数。

In [ ]:
# 我们创建一个需要计算梯度的张量（模拟模型里的一根神经元）
x = torch.tensor([2.0], requires_grad=True)

# 定义一个非常简单的数学运算 y = x^2 + 3x
y = x**2 + 3*x

# 根据微积分，dy/dx = 2x + 3。当 x=2 时，梯度应该是 7
# 让 PyTorch 自动帮我们反向求导
y.backward()

print(f"PyTorch 算出的 x 的梯度 (导数): {x.grad.item()}")
print("🤯 以后所有的模型微调 (Fine-tuning)，本质上就是海量的神经元在做上面这几行代码！")

---
## 兵器二：Processor (处理器与分词器)

大语言模型**根本不认识中文和英文**，它是一台纯血的计算器，只认数字。Tokenizer 的作用就是把我们的长句子“剁碎” (Tokenize) 并翻译成大模型字典里的 ID 数字。

In [ ]:
from transformers import AutoProcessor


print("正在加载 Processor (整合了分词器和图像预处理器)...")
processor = AutoProcessor.from_pretrained(model_id, cache_dir=cache_dir)

# 获取底层的文本分词器看看
tokenizer = processor.tokenizer
print(f"Qwen3.5 词表大小: {tokenizer.vocab_size} 个词块")

In [ ]:
text = "大模型极客实战，从Colab起飞！ LLM is awesome!"

# 编码：人类语言 -> 机器 ID
encoded_ids = tokenizer.encode(text)
print("原本的句子:", text)
print("Token IDs:", encoded_ids)
print(f"文字长度: {len(text)} 字符，Token 数量: {len(encoded_ids)}")

print("\n--- 解剖看看每个 ID 对应什么字 ---")
for tid in encoded_ids:
    print(f"{tid} \t -> \t '{tokenizer.decode([tid])}'")

# 提示：这就是为什么某些英文单词模型怎么也算不对字母个数，因为它脑子里整个单词就是一个数字编码 (例如 awesome)。

---
## 兵器三：量化魔法 (Quantization)

在这个课程里我们要解剖的是 4B 模型 (40亿参数)。

如果我们用默认的纯正高精度浮点数 (FP32，每个数字占 4 个字节) 来加载它，它需要 `4,000,000,000 * 4 Bytes ≈ 16 GB` 的显存。再加上推理时的计算损耗，**免费的 15GB T4 显卡当场就会报 OOM (Out of Memory) 显存溢出错误而崩溃**。

救星就是 **`bitsandbytes`** 库，它能把高精度浮点数压缩成 4 位 (4-bit, 占 0.5个字节) 整数，让模型体积缩小近 8 倍，同时智力几乎不打折扣！

In [ ]:
# 我们先看一下清爽状态下的显卡显存占用情况
!nvidia-smi | grep MiB

In [ ]:
from transformers import AutoModelForImageTextToText, BitsAndBytesConfig
import torch

print("正在使用 4-bit 量化魔法加载多模态架构 (Qwen3.5-4B)...")
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)

# 注意：这是多模态模型的专属类名
model_4bit = AutoModelForImageTextToText.from_pretrained(
    model_id,
    cache_dir=cache_dir,
    device_map="auto",
    quantization_config=quantization_config
)

print("\n✅ 量化加载完成！")

In [ ]:
def print_trainable_parameters(model):
    """
    打印模型参数的数据
    """
    trainable_params = 0
    all_param = 0
    for _, param in model.named_parameters():
        all_param += param.numel()
        if param.requires_grad:
            trainable_params += param.numel()
    print(
        f"全模型参数量: {all_param:,} | 可训练参数量: {trainable_params:,} | "
        f"占比: {100 * trainable_params / all_param:.4f}%"
    )

print_trainable_parameters(model_4bit)
print("你看，因为被 4bit 压缩了，当前模型的所有层都是被冻结（不可计算梯度修饰）的，可训练参数占 0%")
print("\n最后，我们再看一下现在的显存，是不是只有极其可怜的几个 G 的占用？")
!nvidia-smi | grep MiB

---
## 第 1 阶段通关小结
你现在已经拥有了一本厚重的《解剖学图谱》：明白了一切皆是 `Tensor`，懂得了语言是被 `Tokenizer` 切碎的数字，并且学会了像黑客一样用 `4-bit 量化` 把庞然大物硬塞进贫民窟的显卡中。

下一阶段（第 2 阶段），我们将亲手把这只已经被压缩好的 Qwen 剥骨抽筋，找出它进行“逻辑思考”的大脑核心（Attention 和 MLP），甚至我们将暴力切掉它的一个脑叶，看看它会变成什么样！🔥